## Preamble

In [ ]:
# Import packages

import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np
import xarray
import math
import glob
import os
import io
import pathlib
import requests

import rasterio
from rasterio.merge import merge
from rasterio.plot import show
from rasterio.mask import mask
from rasterio.transform import from_bounds
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.enums import Resampling
from rasterio.crs import CRS
from rasterio.features import rasterize

import shapely
from shapely.geometry import Polygon
from shapely.geometry import mapping
from shapely.geometry import box

from tqdm import tqdm
from affine import Affine
from zipfile import ZipFile

import logging
logging.basicConfig(level=logging.INFO)

import atlite
from atlite.gis import ExclusionContainer, shape_availability

In [ ]:
aggregated_regions = snakemake.params.desired_regions

In [ ]:
europe = (
    gpd
    .read_file(snakemake.input.euroshape)
    .query("NUTS_ID == @aggregated_regions")
    .set_index(["NUTS_ID"])
    .loc[:,['geometry']]
) 

In [ ]:
rectx1 = -12
rectx2 = 44
recty1 = 33
recty2 = 72

polygon = Polygon(
    [
        (rectx1, recty1),
        (rectx1, recty2),
        (rectx2, recty2),
        (rectx2, recty1),
        (rectx1, recty1),
    ]
)
europe = gpd.clip(europe, polygon)

## Functions

In [ ]:
def plot_eligible_area(ax, tiff_path, europe, title):
    excluder_wind_onshore = ExclusionContainer()

    full_europe = (
        europe
        .assign(col='a')
        .dissolve(by='col')
        .geometry
    )

    full_europe = full_europe.to_crs(excluder_wind_onshore.crs)

    excluder_wind_onshore.add_raster(tiff_path)
    masked, transform = shape_availability(full_europe, excluder_wind_onshore)
    eligible_share = (masked.sum() * excluder_wind_onshore.res**2 / full_europe.geometry.item().area)
    
    # Plot the eligible area in a subplot
    show(masked, transform=transform, cmap='Greens', ax=ax)
    full_europe.plot(ax=ax, edgecolor='k', color='None')
    europe.to_crs(excluder_wind_onshore.crs).boundary.plot(ax=ax, edgecolor='grey', linewidth=0.2)
    ax.set_title(f'{title} \n Eligible area (green) {eligible_share * 100:2.2f}%')

In [ ]:
def get_bounding():
    
    rectx1 = -12
    rectx2 = 44
    recty1 = 33
    recty2 = 72
    
    polygon = Polygon(
    [
        (rectx1, recty1),
        (rectx1, recty2),
        (rectx2, recty2),
        (rectx2, recty1),
        (rectx1, recty1),
    ]
    )
    
    polygon=shapely.segmentize(polygon, max_segment_length=0.5)
    
    b=gpd.GeoDataFrame(geometry=[polygon],crs="EPSG:4326")
 
    return b.to_crs("EPSG:3035")

In [ ]:
def buffer_and_rasterize_layer(gpkg_file, layer_name, buffer_distance, output_tiff):
    os.makedirs(os.path.dirname(output_tiff), exist_ok=True)
    
    # Read the layer from the GeoPackage
    gdf = gpd.read_file(gpkg_file, layer=layer_name).to_crs(get_bounding().crs)

    # Identify geometry type and buffer lines. 
    # The buffer distance for point data can be adjusted as needed.
    if gdf.geom_type.isin(['LineString', 'MultiLineString','Point']).any():
        # Buffer lines
        resolution = 100
        gdf['geometry'] = gdf['geometry'].buffer(buffer_distance)
    elif gdf.geom_type.isin(['Polygon', 'MultiPolygon']).any():
        resolution = 100
        # Potentially simplify polygons or use them directly if necessary
        gdf['geometry'] = gdf['geometry'].simplify(tolerance=0.001)
        gdf['geometry'] = gdf['geometry'].buffer(buffer_distance)
    else:
        raise ValueError("No suitable geometry types found for processing")

    # Calculate the boundary of the output raster
    #minx, miny, maxx, maxy = gdf.total_bounds
    minx = get_bounding().bounds.minx[0]
    maxx = get_bounding().bounds.maxx[0]
    miny = get_bounding().bounds.miny[0]
    maxy = get_bounding().bounds.maxy[0]
    #resolution = 100  # Adjust resolution as needed
    width = int((maxx - minx) / resolution)
    height = int((maxy - miny) / resolution)
    transform = rasterio.transform.from_bounds(minx, miny, maxx, maxy, width, height)

    # Rasterize the buffered geometries
    rasterized = rasterize(
        [(geom, 1) for geom in gdf.geometry if geom is not None],
        out_shape=(height, width),
        transform=transform,
        fill=0,
        dtype='uint8'
    )

    # Write the raster to file with compression
    with rasterio.open(
        output_tiff, 'w',
        driver='GTiff',
        height=rasterized.shape[0],
        width=rasterized.shape[1],
        count=1,
        dtype='uint8',
        crs='EPSG:3035',
        transform=transform,
        compress='lzw'
    ) as dst:
        dst.write(rasterized, 1)

    print(f'Rasterized buffer saved to {output_tiff}')

In [ ]:
def process_geopackages(base_dir, output_tiff, buffer_poly, buffer_line, resolution=100):
    """
    Fetch the euhydro geopackage files, rasterize and buffer around them. 

    Args:
        base_dir: Directory of all subfolders
        output_tiff: Name of output file
        buffer_poly: Buffer distance for polygons
        buffer_line: Buffer distance for linestrings
        resolution: Resolution of output dataset (default = 100m)
    """
    # Define the pattern to match the main GeoPackage files
    main_file_pattern = "euhydro_*_v013.gpkg"

    # List all matching GeoPackage files
    gpkg_files = [os.path.join(root, file)
                  for root, _, files in os.walk(base_dir)
                  for file in files if fnmatch.fnmatch(file, main_file_pattern)]

    # Layers to include
    layers_to_include = ['InlandWater','River_Net_l','Canals_l']  
    # For CRS in meters
    minx, maxx, miny, maxy = get_bounding().bounds.minx[0], get_bounding().bounds.maxx[0], get_bounding().bounds.miny[0], get_bounding().bounds.maxy[0]

    # Prepare rasterization parameters
    width = int((maxx - minx) / resolution)
    height = int((maxy - miny) / resolution)
    transform = from_bounds(minx, miny, maxx, maxy, width, height)
    raster_merged = None

    # Process each layer and add to raster
    for gpkg_file in tqdm(gpkg_files):
        for layer in layers_to_include:
            raster_layer = buffer_and_rasterize_layer_hydro(gpkg_file, layer, buffer_poly, buffer_line, resolution, transform, (height, width))

            # Continue to the next iteration if raster_layer is None
            if raster_layer is None:
                continue
            
            # Accumulate the rasters, maintaining non-zero values
            if raster_merged is None:
                raster_merged = raster_layer
            else:
                raster_merged = raster_merged | raster_layer

    # Write the combined raster to file with compression
    os.makedirs(os.path.dirname(output_tiff), exist_ok=True)
    with rasterio.open(
        output_tiff, 'w',
        driver='GTiff',
        height=raster_merged.shape[0],
        width=raster_merged.shape[1],
        count=1,
        dtype='uint8',
        crs='EPSG:3035',
        transform=transform,
        compress='lzw'
    ) as dst:
        dst.write(raster_merged, 1)

    print(f'Rasterized buffer saved to {output_tiff}')

In [ ]:
def buffer_and_rasterize_layer_hydro(gpkg_file, layer_name, buffer_poly, buffer_line, resolution, transform, out_shape):
    # Read the layer from the GeoPackage
    gdf = gpd.read_file(gpkg_file, layer=layer_name)
    gdf = gdf.to_crs(get_bounding().crs)
    print(f'processing {gpkg_file} and layer: {layer_name}')
    print("Unique geometry types in the file:", gdf.geom_type.unique())

    # Check if the GeoDataFrame is empty or has unsupported types
    if gdf.empty or gdf.geom_type.isin(['LineString', 'MultiLineString', 'Point', 'Polygon', 'MultiPolygon']).any() is False:
        print(f"Skipping layer {layer_name} in {gpkg_file} due to unsupported or missing geometries.")
        return None
    
    # Buffer geometries based on their type
    if gdf.geom_type.isin(['LineString', 'MultiLineString', 'Point']).any():
        gdf['geometry'] = gdf['geometry'].buffer(buffer_line)
    elif gdf.geom_type.isin(['Polygon', 'MultiPolygon']).any():
        gdf['geometry'] = gdf['geometry'].simplify(tolerance=0.001)
        gdf['geometry'] = gdf['geometry'].buffer(buffer_poly)


    # Rasterize the buffered geometries
    return rasterize(
        [(geom, 1) for geom in gdf.geometry if geom is not None],
        out_shape=out_shape,
        transform=transform,
        fill=0,
        dtype='uint8'
    )

## GeoPackage files (.gpkg)

In [ ]:
powerlines = snakemake.input.powerlinePath
rail = snakemake.input.railPath
airports = snakemake.input.airportPath
glaciers = snakemake.input.glacierPath
roads_motorways = snakemake.input.roads_motorwayPath
roads_primary = snakemake.input.roads_primaryPath
military = snakemake.input.militaryPath
radar = snakemake.input.radarPath
shipping_high = snakemake.input.shippingHighPath
shipping_med = snakemake.input.shippingMedPath


In [ ]:
# Define a dictionary with each key as GeoPackage file and value as list of layers with their buffer distances
buffer_dict = snakemake.params.buffer_dict


In [ ]:
gpkg_files = [
    powerlines,
    rail,
    airports,
    glaciers,
    roads_motorways,
    roads_primary,
    military,
    radar,
    shipping_high,
    shipping_med
]

# Loop over each GeoPackage file and its respective layers
for gpkg_file in tqdm(gpkg_files):
    print(gpkg_file)
    base_name = gpkg_file.split('/')[-1]  # Extract filename
    if base_name in buffer_dict:
        for layer_info in buffer_dict[base_name]:
            layer_name = layer_info['layer_name']
            buffer_distance = layer_info['buffer_distance']
            output_tiff = f'results/individual_exclusions/{base_name.replace(".gpkg", "")}_{layer_name}_buffered.tif'
            buffer_and_rasterize_layer(gpkg_file, layer_name, buffer_distance, output_tiff)



### Rivers

In [ ]:
import warnings
import fnmatch
# Suppress specific warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:
# Example usage
base_directory = snakemake.input.euRiverPath
output_file = snakemake.output.hydrology
buffer_poly = snakemake.params.hydrology_buffer
buffer_line = snakemake.params.hydrology_buffer

process_geopackages(base_directory, output_file, buffer_poly, buffer_line)